# Objectif 

Ce notebook fait suite au transfert des données par Jean-Michel le 02/10/2026. 

L'objectif est pre-process les données. Pour cela l'ensemble des données est converti au format netcdf via xarray. 

Fichiers d'entrée (données brutes): 

* Fichiers de position :
    * Position de l'ATALANTE au cours de la campagne : Trame_GPS_ATL
    * Position de la source lors de chacun des tirs : SHOT (ACOU_ShotPoint001_1.asc)
    * Position AIS : AIS_LPLNE.csv
    * Position des récepteurs : OBS_TLQ_POS.csv


* Fichiers de pression : fichiers binaires des dossiers PRESSURE/OBSi

Fichiers de sortie (après processing) :

* Données de positions : acouplane_pos.nc
* Données de pression : acouplane_raw_pressure.nc

Date de création : 03/10/2026

In [1]:
import os
import sys
import glob
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from scipy.io import loadmat
from datetime import datetime, timedelta

In [2]:
if os.name == "nt":  # Windows
    ROOT_PROGRAM = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd"
else:  # Linux
    ROOT_PROGRAM = "/home/program/ubf_tools"
sys.path.append(ROOT_PROGRAM)

from publication.publication_figure import set_subfigures_abc_labels, color
from real_data_analysis.acouplane.src.acouplane_cst import *
from real_data_analysis.acouplane.src.acouplane_utils import *
from get_data.obs_bin.obs_reader_acouplane import read_obs_data

In [3]:
ensure_dirs_exist()

# Données de position

## Chargement des données 

### Position des récepteurs 

In [4]:
# Read OBS pos
df_rcv_pos = pd.read_csv(
    OBS_TLQ_POS_FPATH,
    dtype={
        "id": str,
        "lon": np.float64,
        "lat": np.float64,
    },
)
df_rcv_pos

,id,lon,lat
0,obs1,5.9526,43.0160
1,obs2,5.9526,43.0170
2,obs4,5.9360,43.0165
3,obs5,5.9930,43.0100
4,obs6,6.0506,42.9440
5,obs7,5.9695,43.0545
6,tlq,5.9695,43.0553


### Trajectoire ATALANTE

In [5]:
files_ATL_mat = sorted(
    glob.glob(
        os.path.join(ACOUPLANE_DATA_POSITION_TRAME_GPS_ATL_DIR, f"*.mat")
    )
)
# print(files_ATL_mat)

time_atl_dt = []
lat_atl_pos = []
lon_atl_pos = []

for fmat in files_ATL_mat:
    atl = loadmat(fmat)["res"]
    # Extract info from mat file
    lat_atl_pos_i, lon_atl_pos_i, time_atl_matlab_i = (
        atl[:, 0],
        atl[:, 1],
        atl[:, 2],
    )
    # Convert time from MATLAB datenum to datetime
    time_atl_dt_i = pd.to_datetime(time_atl_matlab_i - 719529, unit="D", origin="1970-01-01")
    # Drop duplicates if any 
    time_atl_dt_i, idx = np.unique(time_atl_dt_i, return_index=True)
    lon_atl_pos_i = lon_atl_pos_i[idx]
    lat_atl_pos_i = lat_atl_pos_i[idx]
    # print(time_atl_dt_i, lat_atl_pos_i, lon_atl_pos_i)

    # Store 
    time_atl_dt.append(time_atl_dt_i)
    lon_atl_pos.append(lon_atl_pos_i)
    lat_atl_pos.append(lat_atl_pos_i)

# Convert to numpy array
time_atl_dt = np.concat(time_atl_dt)
lon_atl_pos = np.concat(lon_atl_pos)
lat_atl_pos = np.concat(lat_atl_pos)

idx_sort_dt = np.argsort(time_atl_dt)
time_atl_dt = time_atl_dt[idx_sort_dt]
lon_atl_pos = lon_atl_pos[idx_sort_dt]
lat_atl_pos = lat_atl_pos[idx_sort_dt]

# Convert to dataframe
df_atl = pd.DataFrame.from_dict(
    {"datetime": time_atl_dt, "longitude": lon_atl_pos, "latitude": lat_atl_pos},
)
df_atl

,datetime,longitude,latitude
0,2026-02-24 00:00:00.000000000,5.946851,43.028906
1,2026-02-24 00:00:01.000004633,5.946819,43.028906
2,2026-02-24 00:00:01.999999209,5.946787,43.028905
3,2026-02-24 00:00:03.000003842,5.946755,43.028904
4,2026-02-24 00:00:03.999998419,5.946723,43.028903
...,...,...,...
563734,2026-03-02 16:01:02.000002336,5.884835,43.105674
563735,2026-03-02 16:01:02.999996912,5.884835,43.105674
563736,2026-03-02 16:01:04.000001545,5.884835,43.105674
563737,2026-03-02 16:01:04.999996122,5.884835,43.105674


### Données AIS 

In [6]:
def load_ais():
    df_ais = pd.read_csv(
        AIS_FPATH,
        delimiter=";",
        header=0,
        names=["mmsi", "datetime", "lon", "lat"],
    )
    # Convert datatime columns from str to datetime
    fmt = "%d/%m/%Y %H:%M"
    df_ais["datetime"] = pd.to_datetime(df_ais["datetime"], format=fmt)
    # Change dtypes
    df_ais["mmsi"] = df_ais["mmsi"].astype(int)

    # Remove AIS lines with unique positions (at least two points to define a trajectory)
    df_ais = df_ais.groupby("mmsi").filter(lambda x: len(x) > 1)

    # Remove duplicates if any
    df_ais = df_ais.drop_duplicates()

    # Sort 

    return df_ais

In [7]:
def format_ais_data(ais_interp, mmsi, t_start, t_end, time_step):

    t_start = t_start - pd.Timedelta(time_step)
    t_end = t_end + pd.Timedelta(time_step)
    # Define common time vector
    common_time = pd.date_range(start=t_start, end=t_end, freq=time_step)

    ais_lon_mat = []
    ais_lat_mat = []

    for mmsi_i in mmsi:

        ais_mmsi = ais_interp.loc[ais_interp["mmsi"] == mmsi_i]
        # Sort by datetime 
        ais_mmsi = ais_mmsi.sort_values(by="datetime")

        ais_mmsi_arr = ais_mmsi.to_numpy(dtype=np.float64)[
            :, 2:4
        ]  # Exclude time and mmsi column

        # Pad lon lat with NaNs to match common_time length
        try:
            n_pad_start = (ais_mmsi["datetime"].iloc[0] - t_start) // pd.Timedelta(
                time_step
            )
            n_pad_end = (t_end - ais_mmsi["datetime"].iloc[-1]) // pd.Timedelta(time_step)
            ais_mmsi_arr = np.pad(
                ais_mmsi_arr,
                ((n_pad_start, n_pad_end), (0, 0)),
                mode="constant",
                constant_values=np.nan,
            )
        except:
            print(mmsi_i, ais_mmsi)
            continue

        # Add to array
        ais_lon_mat.append(ais_mmsi_arr[:, 0])
        ais_lat_mat.append(ais_mmsi_arr[:, 1])

    ais_lon_mat = np.array(ais_lon_mat)
    ais_lat_mat = np.array(ais_lat_mat)

    return ais_lon_mat, ais_lat_mat, common_time

In [8]:
from source.utils.utils_ais import interpolate_ais
time_step = "10s"
df_ais = load_ais()
df_ais, mmsi = interpolate_ais(ais=df_ais, time_step=time_step)
df_ais

Progress: ████████████████████████████████████████████████████████████████████████████████████████████████████ 100%

,datetime,lon,lat,mmsi
0,2026-02-27 18:23:00,6.372540,43.136820,4
1,2026-02-27 18:23:10,6.372540,43.136820,4
2,2026-02-27 18:23:20,6.372540,43.136820,4
3,2026-02-27 18:23:30,6.372540,43.136820,4
4,2026-02-27 18:23:40,6.372540,43.136820,4
...,...,...,...,...
8140559,2026-03-01 15:06:20,5.759778,43.132174,244
8140560,2026-03-01 15:06:30,5.759594,43.132293,244
8140561,2026-03-01 15:06:40,5.759411,43.132412,244
8140562,2026-03-01 15:06:50,5.759227,43.132531,244


In [9]:
t_start, t_end = df_ais.datetime.min(), df_ais.datetime.max()
print(t_start, t_end)
ais_lon_mat, ais_lat_mat, ais_datetime = format_ais_data(
    ais_interp=df_ais, mmsi=mmsi, t_start=t_start, t_end=t_end, time_step=time_step
)

2026-02-24 00:00:00 2026-03-02 23:54:00


### Positions des tirs sismiques 

In [10]:
files_shot_asc = glob.glob(os.path.join(ACOUPLANE_DATA_POSITION_SHOT_DIR, f"*.asc"))

def asc_to_df(fasc):
    df_shot = pd.read_csv(
        fasc,
        delimiter="\t",
        header=0,
        names=[
            "SEGD",
            "ECOS",
            "date",
            "shot_time",
            "lon_vessel",
            "lat_vessel",
            "lon_source",
            "lat_source",
        ],
        dtype={
            "SEGD": int,
            "ECOS": int,
            "lon_vessel": np.float64,
            "lat_vessel": np.float64,
            "lon_source": np.float64,
            "lat_source": np.float64,
        },
    )

    # Convert date and time to datetime single column
    shot_datetime_str = df_shot["date"] + " " + df_shot["shot_time"]
    fmt = "%d/%m/%Y %H:%M:%S.%f"
    try:
        df_shot["shot_datetime"] = pd.to_datetime(shot_datetime_str, format=fmt)
    except:
        print(fasc)
    # Drop columns
    df_shot = df_shot.drop(columns=["date", "shot_time"])
    return df_shot

df_shot = []

for fasc in files_shot_asc:
    df_shot_i = asc_to_df(fasc)
    df_shot.append(df_shot_i)

df_shot = pd.concat(df_shot)
# Sort by date
df_shot = df_shot.sort_values(by="shot_datetime")
df_shot

,SEGD,ECOS,lon_vessel,lat_vessel,lon_source,lat_source,shot_datetime
0,1,1,6.023298,43.003975,6.024248,43.004038,2026-02-24 20:14:06.600098
1,2,2,6.023214,43.003968,6.024163,43.004031,2026-02-24 20:14:09.600098
2,3,3,6.023129,43.003962,6.024079,43.004024,2026-02-24 20:14:12.600098
3,4,4,6.023044,43.003955,6.023994,43.004017,2026-02-24 20:14:15.600098
4,5,5,6.022960,43.003949,6.023910,43.004010,2026-02-24 20:14:18.600098
...,...,...,...,...,...,...,...
17,19,18,5.994799,43.016170,5.994527,43.015500,2026-02-27 05:28:30.200073
18,20,19,5.994916,43.016951,5.994632,43.016276,2026-02-27 05:29:00.200073
19,21,20,5.995043,43.017730,5.994748,43.017053,2026-02-27 05:29:30.200073
20,22,21,5.995154,43.018508,5.994877,43.017824,2026-02-27 05:30:00.200073


## Merge de l'ensemble des données 

In [11]:
df_rcv_pos

,id,lon,lat
0,obs1,5.9526,43.0160
1,obs2,5.9526,43.0170
2,obs4,5.9360,43.0165
3,obs5,5.9930,43.0100
4,obs6,6.0506,42.9440
5,obs7,5.9695,43.0545
6,tlq,5.9695,43.0553


In [12]:
df_atl.dtypes, df_ais.dtypes, df_shot.dtypes

(datetime     datetime64[ns]
 longitude           float64
 latitude            float64
 dtype: object,
 datetime    datetime64[us]
 lon                float64
 lat                float64
 mmsi                 int64
 dtype: object,
 SEGD                      int64
 ECOS                      int64
 lon_vessel              float64
 lat_vessel              float64
 lon_source              float64
 lat_source              float64
 shot_datetime    datetime64[us]
 dtype: object)

In [13]:
df_ais.mmsi

0            4
1            4
2            4
3            4
4            4
          ... 
8140559    244
8140560    244
8140561    244
8140562    244
8140563    244
Name: mmsi, Length: 8140564, dtype: int64

In [14]:
ds_pos = xr.Dataset(
    data_vars=dict(
        # RCV pos
        rcv_lon=(["rcv_id"], df_rcv_pos.lon),
        rcv_lat=(["rcv_id"], df_rcv_pos.lat),
        # ATL pos
        atl_lon=(["atl_datetime"], df_atl.longitude),
        atl_lat=(["atl_datetime"], df_atl.latitude),
        # AIS pos
        ais_lon=(["mmsi", "ais_datetime"], ais_lon_mat),
        ais_lat=(["mmsi", "ais_datetime"], ais_lat_mat),
        # Tirs source
        tir_SEGD=(["src_datetime"], df_shot.SEGD),
        tir_ECOS=(["src_datetime"], df_shot.ECOS),
        tir_src_lon=(["src_datetime"], df_shot.lon_source),
        tir_src_lat=(["src_datetime"], df_shot.lat_source),
        tir_vessel_lon=(["src_datetime"], df_shot.lon_vessel),
        tir_vessel_lat=(["src_datetime"], df_shot.lat_vessel),
    ),
    coords=dict(
        rcv_id=df_rcv_pos.id.values,
        ais_datetime=ais_datetime,
        atl_datetime=df_atl.datetime,
        src_datetime=df_shot.shot_datetime,
    ),
    attrs=dict(
        description="Positions AIS interpolées",
        interpolation_time_step=time_step,
        campagne="ACOUPLANE",
        geodesic_frame="WGS84",
    ),
)

In [15]:
# Add attributes to variables
ds_pos.rcv_lon.attrs["units"] = "°"
ds_pos.rcv_lat.attrs["units"] = "°"
ds_pos.atl_lon.attrs["units"] = "°"
ds_pos.atl_lat.attrs["units"] = "°"
ds_pos.ais_lon.attrs["units"] = "°"
ds_pos.ais_lat.attrs["units"] = "°"
ds_pos.tir_src_lon.attrs["units"] = "°"
ds_pos.tir_src_lat.attrs["units"] = "°"
ds_pos.tir_vessel_lon.attrs["units"] = "°"
ds_pos.tir_vessel_lat.attrs["units"] = "°"

ds_pos.rcv_lon.attrs["long_name"] = "Longitude"
ds_pos.rcv_lat.attrs["long_name"] = "Latitude"
ds_pos.atl_lon.attrs["long_name"] = "Longitude"
ds_pos.atl_lat.attrs["long_name"] = "Latitude"
ds_pos.ais_lon.attrs["long_name"] = "Longitude"
ds_pos.ais_lat.attrs["long_name"] = "Latitude"
ds_pos.tir_src_lon.attrs["long_name"] = "Longitude"
ds_pos.tir_src_lat.attrs["long_name"] = "Latitude"
ds_pos.tir_vessel_lon.attrs["long_name"] = "Longitude"
ds_pos.tir_vessel_lat.attrs["long_name"] = "Latitude"


In [16]:
ds_pos

<xarray.Dataset> Size: 525MB
Dimensions:         (rcv_id: 7, atl_datetime: 563739, mmsi: 526,
                     ais_datetime: 60447, src_datetime: 33608)
Coordinates:
  * rcv_id          (rcv_id) object 56B 'obs1' 'obs2' 'obs4' ... 'obs7' 'tlq'
  * atl_datetime    (atl_datetime) datetime64[ns] 5MB 2026-02-24 ... 2026-03-...
  * ais_datetime    (ais_datetime) datetime64[us] 484kB 2026-02-23T23:59:50 ....
  * src_datetime    (src_datetime) datetime64[us] 269kB 2026-02-24T20:14:06.6...
Dimensions without coordinates: mmsi
Data variables:
    rcv_lon         (rcv_id) float64 56B 5.953 5.953 5.936 5.993 6.051 5.97 5.97
    rcv_lat         (rcv_id) float64 56B 43.02 43.02 43.02 ... 42.94 43.05 43.06
    atl_lon         (atl_datetime) float64 5MB 5.947 5.947 5.947 ... 5.885 5.885
    atl_lat         (atl_datetime) float64 5MB 43.03 43.03 43.03 ... 43.11 43.11
    ais_lon         (mmsi, ais_datetime) float64 254MB nan nan nan ... nan nan
    ais_lat         (mmsi, ais_datetime) float64 254MB nan nan nan ... nan nan
    tir_SEGD        (src_datetime) int64 269kB 1 2 3 4 5 6 ... 18 19 20 21 22 23
    tir_ECOS        (src_datetime) int64 269kB 1 2 3 4 5 6 ... 17 18 19 20 21 22
    tir_src_lon     (src_datetime) float64 269kB 6.024 6.024 ... 5.995 5.995
    tir_src_lat     (src_datetime) float64 269kB 43.0 43.0 43.0 ... 43.02 43.02
    tir_vessel_lon  (src_datetime) float64 269kB 6.023 6.023 ... 5.995 5.995
    tir_vessel_lat  (src_datetime) float64 269kB 43.0 43.0 43.0 ... 43.02 43.02
Attributes:
    description:              Positions AIS interpolées
    interpolation_time_step:  10s
    campagne:                 ACOUPLANE
    geodesic_frame:           WGS84

In [19]:
ds_pos.to_netcdf(ACOUPLANE_POSITIONS_NC_FPATH)

# Données de pression

## Chargement des données 

In [20]:
# Read all OBS data from a main folder containing subfolders for each OBS
# data_folder = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\data\ACOUPLANE\DATA\PRESSURE\BIN"
obs_data, obs_time, fullScale = read_obs_data(
    data_folder=ACOUPLANE_DATA_PRESSURE_BIN_DIR, read_raw=True, verbose=False
)

Reading OBS data from folder: /home/data/ACOUPLANE/DATA/PRESSURE/BIN/OBS1
Reading file: /home/data/ACOUPLANE/DATA/PRESSURE/BIN/OBS1/ELOBS_D-2744775_TB_1455975618000000-1456000620499500_channel3_2026-03-02-08-45-30_002.bin
Recording from 24/02/2026 13:40:00 UTC to 24/02/2026 20:36:42 UTC
Reading file: /home/data/ACOUPLANE/DATA/PRESSURE/BIN/OBS1/ELOBS_D-2744775_TB_1456000620500000-1456025622499500_channel3_2026-03-02-08-45-30_002.bin
Recording from 24/02/2026 20:36:42 UTC to 25/02/2026 03:33:24 UTC
Reading file: /home/data/ACOUPLANE/DATA/PRESSURE/BIN/OBS1/ELOBS_D-2744775_TB_1456025622500000-1456050626249500_channel3_2026-03-02-08-45-30_002.bin
Recording from 25/02/2026 03:33:24 UTC to 25/02/2026 10:30:08 UTC
Reading file: /home/data/ACOUPLANE/DATA/PRESSURE/BIN/OBS1/ELOBS_D-2744775_TB_1456050626250000-1456075631249500_channel3_2026-03-02-08-45-30_002.bin
Recording from 25/02/2026 10:30:08 UTC to 25/02/2026 17:26:53 UTC
Reading file: /home/data/ACOUPLANE/DATA/PRESSURE/BIN/OBS1/ELOBS_D-2744

In [21]:
obs_data

{'OBS1': array([26290944, 25813504, 26752768, ..., 20817920, 25030912, 29129984],
       shape=(995998000,), dtype=int32),
 'OBS2': array([36745216, 27983360, 17442048, ..., 27738624, 27982080, 23679232],
       shape=(1000246000,), dtype=int32),
 'OBS4': array([21543424, 25940736, 31615232, ..., 26055936, 27102976, 25680896],
       shape=(993598000,), dtype=int32),
 'OBS5': array([28526848, 27170560, 26589184, ..., 27037184, 28952832, 27178240],
       shape=(831598000,), dtype=int32),
 'OBS6': array([24790016, 30931712, 29457920, ..., 30603008, 32458240, 31345152],
       shape=(818998000,), dtype=int32),
 'OBS7': array([26481152, 25143296, 26538496, ..., 24093440, 32775168, 34028544],
       shape=(850176000,), dtype=int32)}

In [22]:
obs_time

{'OBS1': array(['2026-02-24T13:40:00.000000', '2026-02-24T13:40:00.000500',
        '2026-02-24T13:40:00.001000', ..., '2026-03-02T07:59:58.998500',
        '2026-03-02T07:59:58.999000', '2026-03-02T07:59:58.999500'],
       shape=(995998000,), dtype='datetime64[us]'),
 'OBS2': array(['2026-02-24T12:45:00.000000', '2026-02-24T12:45:00.000500',
        '2026-02-24T12:45:00.001000', ..., '2026-03-02T07:40:22.998500',
        '2026-03-02T07:40:22.999000', '2026-03-02T07:40:22.999500'],
       shape=(1000246000,), dtype='datetime64[us]'),
 'OBS4': array(['2026-02-24T13:00:00.000000', '2026-02-24T13:00:00.000500',
        '2026-02-24T13:00:00.001000', ..., '2026-03-02T06:59:58.998500',
        '2026-03-02T06:59:58.999000', '2026-03-02T06:59:58.999500'],
       shape=(993598000,), dtype='datetime64[us]'),
 'OBS5': array(['2026-02-24T16:00:00.000000', '2026-02-24T16:00:00.000500',
        '2026-02-24T16:00:00.001000', ..., '2026-03-01T11:29:58.998500',
        '2026-03-01T11:29:58.999000', '2

## Merge des données 

Pour éviter de stocker des données inutilement lourdes il faut : 

* stocker la donnée dans le format d'origine int32
* ne pas stocker le vecteur de datetimes mais seulement la date de début et la fréquence d'échantillonnage 

In [23]:
data_vars = {
    f"pressure_{obs_id.lower()}": ([f"time_{obs_id.lower()}"], obs_data[obs_id])
    for obs_id in obs_data.keys()
}
data_vars

{'pressure_obs1': (['time_obs1'],
  array([26290944, 25813504, 26752768, ..., 20817920, 25030912, 29129984],
        shape=(995998000,), dtype=int32)),
 'pressure_obs2': (['time_obs2'],
  array([36745216, 27983360, 17442048, ..., 27738624, 27982080, 23679232],
        shape=(1000246000,), dtype=int32)),
 'pressure_obs4': (['time_obs4'],
  array([21543424, 25940736, 31615232, ..., 26055936, 27102976, 25680896],
        shape=(993598000,), dtype=int32)),
 'pressure_obs5': (['time_obs5'],
  array([28526848, 27170560, 26589184, ..., 27037184, 28952832, 27178240],
        shape=(831598000,), dtype=int32)),
 'pressure_obs6': (['time_obs6'],
  array([24790016, 30931712, 29457920, ..., 30603008, 32458240, 31345152],
        shape=(818998000,), dtype=int32)),
 'pressure_obs7': (['time_obs7'],
  array([26481152, 25143296, 26538496, ..., 24093440, 32775168, 34028544],
        shape=(850176000,), dtype=int32))}

In [24]:
# Add start datetime for each OBS
data_vars.update(
    {
        "start_datetime": (
            ["obs_id"],
            [obs_time[obs_id][0] for obs_id in obs_time.keys()],
        )
    }
)

# data_vars.update(
#     {
#         f"start_datetime_{obs_id.lower()}": (
#             [],
#             obs_time[obs_id][0],
#         )
#         for obs_id in obs_time.keys()
#     }
# )
data_vars

{'pressure_obs1': (['time_obs1'],
  array([26290944, 25813504, 26752768, ..., 20817920, 25030912, 29129984],
        shape=(995998000,), dtype=int32)),
 'pressure_obs2': (['time_obs2'],
  array([36745216, 27983360, 17442048, ..., 27738624, 27982080, 23679232],
        shape=(1000246000,), dtype=int32)),
 'pressure_obs4': (['time_obs4'],
  array([21543424, 25940736, 31615232, ..., 26055936, 27102976, 25680896],
        shape=(993598000,), dtype=int32)),
 'pressure_obs5': (['time_obs5'],
  array([28526848, 27170560, 26589184, ..., 27037184, 28952832, 27178240],
        shape=(831598000,), dtype=int32)),
 'pressure_obs6': (['time_obs6'],
  array([24790016, 30931712, 29457920, ..., 30603008, 32458240, 31345152],
        shape=(818998000,), dtype=int32)),
 'pressure_obs7': (['time_obs7'],
  array([26481152, 25143296, 26538496, ..., 24093440, 32775168, 34028544],
        shape=(850176000,), dtype=int32)),
 'start_datetime': (['obs_id'],
  [np.datetime64('2026-02-24T13:40:00.000000'),
   np.d

In [25]:
data_vars["pressure_obs1"][1].dtype

dtype('int32')

In [26]:
# Build dataset
ds_pressure = xr.Dataset(
    data_vars=data_vars,
    # coords={f"time_{obs_id.lower()}": obs_time[obs_id] for obs_id in obs_data.keys()},
    coords=dict(obs_id=[obs_id.lower() for obs_id in obs_data.keys()]),
    attrs=dict(
        description="Données de pression (voie hydro 3)",
        campagne="ACOUPLANE",
        fs=2000,
        fullscale=fullScale,
    ),
)
# Add attributes to variables
for obs_id in ds_pressure.obs_id.values:
    ds_pressure[f"pressure_{obs_id}"].attrs["units"] = ""   # No units 
    ds_pressure[f"pressure_{obs_id}"].attrs["long_name"] = "Pressure"   
    # ds_pressure[f"pressure_{obs_id}"].attrs["long_name"] = f"Pressure ({obs_id.upper()})"   

    # ds_pressure[f"time_{obs_id}"].attrs["units"] = "UTC"   # No units 
    # ds_pressure[f"time_{obs_id}"].attrs["long_name"] = f"Time ({obs_id.upper()})"

In [27]:
ds_pressure

<xarray.Dataset> Size: 22GB
Dimensions:         (time_obs1: 995998000, time_obs2: 1000246000,
                     time_obs4: 993598000, time_obs5: 831598000,
                     time_obs6: 818998000, time_obs7: 850176000, obs_id: 6)
Coordinates:
  * obs_id          (obs_id) <U4 96B 'obs1' 'obs2' 'obs4' 'obs5' 'obs6' 'obs7'
Dimensions without coordinates: time_obs1, time_obs2, time_obs4, time_obs5,
                                time_obs6, time_obs7
Data variables:
    pressure_obs1   (time_obs1) int32 4GB 26290944 25813504 ... 29129984
    pressure_obs2   (time_obs2) int32 4GB 36745216 27983360 ... 23679232
    pressure_obs4   (time_obs4) int32 4GB 21543424 25940736 ... 25680896
    pressure_obs5   (time_obs5) int32 3GB 28526848 27170560 ... 27178240
    pressure_obs6   (time_obs6) int32 3GB 24790016 30931712 ... 31345152
    pressure_obs7   (time_obs7) int32 3GB 26481152 25143296 ... 34028544
    start_datetime  (obs_id) datetime64[us] 48B 2026-02-24T13:40:00 ... 2026-...
Attributes:
    description:  Données de pression (voie hydro 3)
    campagne:     ACOUPLANE
    fs:           2000
    fullscale:    4.52

In [28]:
ds_pressure.to_netcdf(ACOUPLANE_PRESSURE_NC_FPATH)